# Bài học 10 - Các tác nhân AI trong môi trường sản xuất

Trong bài học này bạn sẽ học **các mẫu triển khai trong môi trường sản xuất** cho các tác nhân AI sử dụng Microsoft Agent Framework (Python). Chúng ta sẽ đề cập đến:

- **Khả năng quan sát (Observability)** — thêm theo dõi thời gian và ghi nhật ký vào các tương tác của tác nhân
- **Đánh giá (Evaluation)** — sử dụng một tác nhân đánh giá để chấm điểm chất lượng phản hồi
- **Quản lý chi phí (Cost management)** — chiến lược tối ưu hóa token và lựa chọn mô hình

Kịch bản là một **đại lý du lịch** giúp người dùng lên kế hoạch cho chuyến đi, với giám sát và đánh giá được tích hợp thêm.


## Cài đặt


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Production Considerations

Moving AI agents from prototypes to production requires careful attention to three pillars:

1. **Quan sát (Observability)** — Bạn cần thấy được tác nhân đang làm gì, mất bao lâu và nó gọi những công cụ nào. Nếu không có theo dõi và ghi nhật ký, việc gỡ lỗi các sự cố trong môi trường sản xuất hầu như không thể thực hiện.

2. **Đánh giá (Evaluation)** — Các kiểm tra chất lượng tự động đảm bảo các phản hồi của tác nhân duy trì tính chính xác, đầy đủ và hữu ích theo thời gian. Một tác nhân đánh giá có thể chấm điểm các phản hồi dựa trên các tiêu chí đã định nghĩa.

3. **Quản lý chi phí (Cost Management)** — Việc sử dụng token ảnh hưởng trực tiếp đến chi phí. Các chiến lược như tối ưu prompt, lựa chọn mô hình và caching giúp giữ chi phí trong tầm kiểm soát mà không làm giảm chất lượng.


## Xây dựng một Agent có thể quan sát

Chúng tôi định nghĩa các công cụ du lịch và bao bọc cuộc gọi tới agent bằng việc ghi thời gian để chúng ta có thể giám sát độ trễ. Trong môi trường sản xuất, bạn sẽ tích hợp với OpenTelemetry hoặc một backend theo dõi (tracing) tương tự.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Các mẫu đánh giá

Một mô hình triển khai phổ biến là sử dụng một tác nhân thứ hai làm **người đánh giá**. Người đánh giá chấm điểm phản hồi của tác nhân chính dựa trên các tiêu chí đã định trước như tính đầy đủ, độ chính xác và tính hữu ích.

Điều này cho phép:
- Các cổng kiểm soát chất lượng tự động trước khi phản hồi đến người dùng
- Phát hiện suy giảm khi lời nhắc hoặc mô hình thay đổi
- Giám sát liên tục hiệu suất của tác nhân theo thời gian


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Chiến lược Quản lý Chi phí

Kiểm soát chi phí là điều then chốt cho các tác nhân AI trong môi trường sản xuất. Dưới đây là các chiến lược chính:

| Strategy | Description |
|---|---|
| **Tối ưu lời nhắc** | Giữ các hướng dẫn hệ thống ngắn gọn. Loại bỏ ngữ cảnh thừa để giảm số token đầu vào. |
| **Lựa chọn mô hình** | Sử dụng các mô hình nhỏ hơn, rẻ hơn (ví dụ GPT-4o-mini) cho các tác vụ đơn giản như phân loại hoặc trích xuất, và dành các mô hình lớn hơn cho suy luận phức tạp. |
| **Bộ nhớ đệm** | Lưu kết quả công cụ và các truy vấn thường xuyên để tránh các lần gọi API bị lặp lại. |
| **Ngân sách token** | Đặt giới hạn `max_tokens` để ngăn chặn phản hồi dài ngoài mong đợi. |
| **Gộp theo lô** | Gộp nhiều truy vấn của người dùng vào một lần gọi API khi có thể. |

Trong thực tế, một cách tiếp cận theo tầng hoạt động tốt: định tuyến các yêu cầu đơn giản đến một mô hình nhanh và rẻ, và chỉ nâng cấp những truy vấn phức tạp hơn đến một mô hình có khả năng mạnh hơn.


## Tóm tắt

Trong bài học này bạn đã học cách:

1. **Thêm khả năng quan sát** vào tương tác của tác nhân bằng việc đo thời gian và ghi nhật ký, đặt nền tảng cho truy vết và giám sát.
2. **Đánh giá phản hồi của tác nhân** tự động bằng một tác nhân đánh giá chấm điểm độ đầy đủ, độ chính xác và tính hữu ích.
3. **Quản lý chi phí** thông qua tối ưu hóa lời nhắc, lựa chọn mô hình, bộ nhớ đệm và ngân sách token.

Những mô hình triển khai này giúp đảm bảo các tác nhân AI của bạn đáng tin cậy, có thể đo lường và tiết kiệm chi phí khi mở rộng quy mô.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Miễn trừ trách nhiệm:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng các bản dịch tự động có thể chứa lỗi hoặc thông tin không chính xác. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn có thẩm quyền. Đối với các thông tin quan trọng, khuyến nghị sử dụng dịch vụ dịch thuật chuyên nghiệp do con người thực hiện. Chúng tôi không chịu trách nhiệm về bất kỳ sự hiểu lầm hay diễn giải sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
